In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/2일차'
SYSTEM = open(f'{DATA}/분류_규칙.txt', encoding='utf-8').read()

# 100만 토큰당 달러 (입력, 출력). Anthropic 공개 가격, 2026-11-03 확인.
# 사내 API 를 쓴다면 사내 과금 기준으로 바꿉니다. 과금이 없으면 (0, 0) 으로 둡니다.
PRICE = {'소형': (0.10, 0.50), '기본': (2.00, 10.00), '대형': (4.00, 20.00)}


def cost_usd(input_tokens, output_tokens, model='기본'):
    """토큰 수와 단가로 요금(달러)을 계산합니다."""
    input_price, output_price = PRICE[model]
    return (input_tokens * input_price + output_tokens * output_price) / 1_000_000


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


In [3]:
# 실습 준비
import sqlite3
from typing import Literal

from pydantic import BaseModel, ConfigDict, ValidationError

import checks2

tickets = pd.read_csv(f'{DATA}/접수_실습.csv').fillna('')
ticket_answers = pd.read_csv(f'{DATA}/접수_실습_정답.csv', dtype=str, keep_default_na=False).set_index('id')
ticket_answers['urgent'] = ticket_answers['urgent'] == 'true'
ticket_answers['location'] = ticket_answers['location'].map(lambda value: value or None)
TICKET_RULES = open(f'{DATA}/접수_규칙.txt', encoding='utf-8').read().strip()
print(TICKET_RULES)
print('----')
print(tickets.head(8).to_string(index=False))


def prompt_of(row):
    """접수 한 건을 모델에 보낼 글로 만듭니다."""
    return f"id: {row['id']}\n본문: {row['본문']}"


def parse_reply(reply, model_class):
    """응답을 검증해 (객체, None) 또는 (None, 사유) 를 돌려줍니다."""
    if reply.stop_reason != 'end':
        return None, {'max_tokens': '잘림', 'refusal': '거절'}.get(reply.stop_reason, '비정상 종료')
    text = reply.text[reply.text.find('{'):reply.text.rfind('}') + 1]
    try:
        return model_class.model_validate_json(text), None
    except ValidationError as error:
        first = error.errors()[0]
        field = '.'.join(str(part) for part in first['loc']) or '전체'
        return None, f"스키마 불일치({field}: {first['type']})"


def open_db(path=':memory:'):
    """requests 테이블이 있는 DB 를 엽니다. 경로를 주지 않으면 메모리에만 있는 DB 입니다."""
    db = sqlite3.connect(path)
    db.execute('''CREATE TABLE IF NOT EXISTS requests (
        id TEXT PRIMARY KEY, team TEXT, location TEXT, urgent INTEGER,
        model TEXT, run_at TEXT, reviewed TEXT)''')
    return db

너는 반도체 공장에 들어오는 접수 요청을 담당팀으로 나누는 도우미다.

담당팀(team)은 아래 다섯 가지 중 하나다.
- 설비: 생산 장비의 고장, 알람, 점검, 부품 교체
- 시설: 건물, 공조, 전기, 수도, 조명, 사무 집기와 편의 시설. 초순수와 전력 같은 유틸리티 공급도 시설이다
- IT: PC, 계정, 네트워크, 서버, 사내 시스템, 프린터
- 안전: 사람의 안전, 화학물질과 가스 누출, 보호구, 화재, 출입 보안
- 기타: 위 네 팀의 일이 아닌 요청이나 요청이 아닌 글

두 팀에 걸치면 안전, 설비, 시설, IT 순으로 앞선 것을 고른다.
location 은 본문에 나온 장비 이름이나 장소를 본문의 글자 그대로 옮긴 것이다. 본문에 없으면 null 이다.
urgent 는 생산이 멈췄거나 사람이 다칠 수 있는 상황이면 true, 아니면 false 다.
----
  id             접수시각                                                본문
R001 2026-11-09 01:14                        통근 버스 노선에 정류장을 하나 추가해 주세요.
R002 2026-11-09 02:28                       다음 달 연차 사용 계획을 어디에 올리면 되나요.
R003 2026-11-09 04:20                              오늘 회식 장소 아시는 분 계신가요.
R004 2026-11-09 05:00        SPT-03 타깃 수명이 90%를 넘었습니다. 교체 계획 확인 부탁드립니다.
R005 2026-11-09 06:22      엘리베이터 2호기가 3층에서 멈춘 채 움직이지 않습니다. 안에 사람은 없습니다.
R006 2026-11-09 06:37                     출입 카드를 잃어버렸습니다. 정지 처리 부탁드립니다.
R007 2026-11-09 10:40                               볼펜과 포

In [4]:
# 실습 5
class Ticket(BaseModel):
    """접수 한 건의 결과. 필드는 id, team, location, urgent 입니다."""
    model_config = ConfigDict(extra='forbid', strict=True)
    id: str
    team: Literal['설비', '시설', 'IT', '안전', '기타']
    location: str | None          # 기본값을 주지 않습니다. 없으면 null 이 와야 합니다
    urgent: bool

In [5]:
# 실습 6
def classify(row, ask=llm.ask, model_class=None, rules=None):
    """접수 한 건을 보내 결과 dict 를 돌려줍니다. 실패해도 예외를 내지 않습니다."""
    model_class = model_class or Ticket
    record = {'id': row['id'], 'answer_id': None, 'team': None, 'location': None, 'urgent': None,
              'error': None, 'input_tokens': 0, 'output_tokens': 0}
    try:
        reply = ask(prompt_of(row), system=rules or TICKET_RULES, model='소형', max_tokens=1024,
                    schema=model_class.model_json_schema())
    except llm.CallError as error:            # API 호출의 실패만 잡습니다
        record['error'] = f'호출 실패({error.kind})'
        return record
    record.update(input_tokens=reply.input_tokens or 0, output_tokens=reply.output_tokens or 0)
    parsed, reason = parse_reply(reply, model_class)
    if reason:
        record['error'] = reason
    else:
        record.update(answer_id=parsed.id, team=parsed.team, location=parsed.location, urgent=parsed.urgent)
    return record


UPSERT = '''INSERT INTO requests (id, team, location, urgent, model, run_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(id) DO UPDATE SET
                team = excluded.team, location = excluded.location, urgent = excluded.urgent,
                model = excluded.model, run_at = excluded.run_at'''


def save(db, records, model, run_at):
    """error 가 없는 결과를 requests 테이블에 넣고 넣은 건수를 돌려줍니다. 같은 id 는 결과 열만 고칩니다."""
    db_rows = [(record['id'], record['team'], record['location'], int(record['urgent']), model, run_at)
               for record in records if not record['error']]
    db.executemany(UPSERT, db_rows)
    db.commit()
    return len(db_rows)


def run(db, rows, ask=llm.ask):
    """requests 테이블에 없는 id 만 보내고 저장합니다. (보낸 건수, 실패 목록) 을 돌려줍니다."""
    done_ids = {row_id for (row_id,) in db.execute('SELECT id FROM requests')}
    to_send = [row for row in rows if row['id'] not in done_ids]
    with ThreadPoolExecutor(max_workers=8) as pool:
        records = list(pool.map(lambda row: classify(row, ask=ask), to_send))
    save(db, records, llm.model_name('소형'), pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'))
    return len(to_send), [(record['id'], record['error']) for record in records if record['error']]

In [6]:
# 실습 7
def validate(record, text):
    """결과 한 건을 검사해 걸린 사유를, 통과하면 None 을 돌려줍니다."""
    if record['error']:
        return record['error']
    if record['answer_id'] != record['id']:
        return 'id 불일치'
    if record['team'] == '기타' and record['urgent']:
        return '기타인데 urgent 가 true'
    if record['location'] is not None and record['location'] not in text:
        return '본문에 없는 위치'
    return None


def status_of(record, text):
    """'auto', 'review', 'manual' 가운데 하나를 돌려줍니다."""
    if record['error']:
        return 'manual'                    # 모델의 결과가 없습니다
    return 'review' if validate(record, text) else 'auto'

In [7]:
# 실습 7 확인 (API 를 부르지 않습니다)
checks2.check_validate(validate, status_of)

1. 통과 문제가 없는 결과는 통과한다
2. 통과 위치가 None 인 결과는 통과한다
3. 통과 error 가 있으면 그 사유를 돌려준다
4. 통과 보낸 id 와 받은 id 가 다르면 걸린다
5. 통과 기타인데 urgent 가 true 면 걸린다
6. 통과 본문에 없는 위치는 걸린다
실습 7 통과


True

In [8]:
# 실습 7 측정 (실제 API 를 부릅니다. 80건)
rows = [row for _, row in tickets.iterrows()]
text_of = dict(zip(tickets['id'], tickets['본문']))


def run_all(model_class=None, rules=None):
    """80건을 모두 분류해 결과 dict 의 목록을 돌려줍니다. DB 에는 저장하지 않습니다."""
    with ThreadPoolExecutor(max_workers=8) as pool:
        return list(pool.map(lambda row: classify(row, model_class=model_class, rules=rules), rows))


def report(records, label):
    """검증 결과와 정답 라벨 비교를 출력합니다."""
    done = [record for record in records if not record['error']]
    team_ok = sum(record['team'] == ticket_answers.loc[record['id'], 'team'] for record in done)
    urgent_ok = sum(record['urgent'] == ticket_answers.loc[record['id'], 'urgent'] for record in done)
    statuses = [status_of(record, text_of[record['id']]) for record in records]
    reasons = pd.Series([validate(record, text_of[record['id']]) for record in records]).dropna()
    auto_wrong = [record['id'] for record, status in zip(records, statuses)
                  if status == 'auto' and record['team'] != ticket_answers.loc[record['id'], 'team']]
    print(f'[{label}] 스키마 검증 통과 {len(done)}/{len(records)}'
          f' · 담당팀 정확도 {team_ok}/{len(done)} · 긴급 여부 정확도 {urgent_ok}/{len(done)}')
    print('  상태:', pd.Series(statuses).value_counts().to_dict(), '· 걸린 사유:', reasons.value_counts().to_dict())
    print(f'  auto {statuses.count("auto")}건 가운데 담당팀이 틀린 건 {len(auto_wrong)}건', auto_wrong)
    print('  urgent 가 true 인 건', sum(bool(record['urgent']) for record in done),
          f"(정답은 {int(ticket_answers['urgent'].sum())}건)")


base = run_all()
report(base, '실험 전')
for record in base:
    answer = ticket_answers.loc[record['id']]
    why = validate(record, text_of[record['id']])
    if why or record['team'] != answer['team']:
        print(f"  {record['id']} 모델 {record['team']} / {record['location']!r} · 정답 {answer['team']}"
              f" · 사유 {why} · {text_of[record['id']][:30]}")

[실험 전] 스키마 검증 통과 80/80 · 담당팀 정확도 76/80 · 긴급 여부 정확도 79/80
  상태: {'auto': 80} · 걸린 사유: {}
  auto 80건 가운데 담당팀이 틀린 건 4건 ['R022', 'R026', 'R036', 'R073']
  urgent 가 true 인 건 22 (정답은 21건)
  R022 모델 설비 / '클린룸' · 정답 시설 · 사유 None · 클린룸 차압이 기준 아래로 내려갔다는 경보가 떴습니다.
  R026 모델 시설 / '클린룸 입구 키오스크' · 정답 IT · 사유 None · 클린룸 입구 키오스크 화면이 꺼져 있습니다. 급하지는 
  R036 모델 설비 / 'B라인' · 정답 시설 · 사유 None · 방금 순간 정전이 있었고 B라인 장비가 전부 꺼졌습니다
  R073 모델 안전 / '세정 장비 3호기' · 정답 설비 · 사유 None · 세정 장비 3호기 배수 밸브가 닫히지 않습니다. 약액이


In [9]:
# 실습 7 실험 A (담당팀 허용값에서 IT 를 뺍니다. 80건을 다시 보냅니다)
class TicketNoIT(Ticket):
    team: Literal['설비', '시설', '안전', '기타']


no_it = run_all(model_class=TicketNoIT)
report(no_it, 'IT 제외')
it_ids = set(ticket_answers.index[ticket_answers['team'] == 'IT'])
moved = pd.Series([record['team'] for record in no_it if record['id'] in it_ids])
print(f'  정답이 IT 인 {len(it_ids)}건의 분류:', moved.value_counts(dropna=False).to_dict())

[IT 제외] 스키마 검증 통과 80/80 · 담당팀 정확도 64/80 · 긴급 여부 정확도 80/80
  상태: {'auto': 79, 'review': 1} · 걸린 사유: {'기타인데 urgent 가 true': 1}
  auto 79건 가운데 담당팀이 틀린 건 15건 ['R023', 'R026', 'R030', 'R035', 'R041', 'R042', 'R044', 'R047', 'R053', 'R057', 'R058', 'R063', 'R067', 'R070', 'R078']
  urgent 가 true 인 건 21 (정답은 21건)
  정답이 IT 인 16건의 분류: {'기타': 12, '시설': 4}


In [10]:
# 실습 7 실험 B (규칙에서 긴급 기준 문장을 뺍니다. 80건을 다시 보냅니다)
URGENT_LINE = next(line for line in TICKET_RULES.splitlines() if line.startswith('urgent'))
print('뺀 문장:', URGENT_LINE)
no_rule = run_all(rules=TICKET_RULES.replace(URGENT_LINE, '').strip())
report(no_rule, '긴급 기준 제외')
flipped = [(new, old) for new, old in zip(no_rule, base) if new['urgent'] != old['urgent']]
print('  긴급 판단이 바뀐 건', len(flipped))
for new, old in flipped[:10]:
    print(f"  {new['id']} 실험 전 {old['urgent']} → {new['urgent']}"
          f" · 정답 {ticket_answers.loc[new['id'], 'urgent']} · {text_of[new['id']][:34]}")

뺀 문장: urgent 는 생산이 멈췄거나 사람이 다칠 수 있는 상황이면 true, 아니면 false 다.


[긴급 기준 제외] 스키마 검증 통과 80/80 · 담당팀 정확도 77/80 · 긴급 여부 정확도 74/80
  상태: {'auto': 80} · 걸린 사유: {}
  auto 80건 가운데 담당팀이 틀린 건 3건 ['R036', 'R038', 'R073']
  urgent 가 true 인 건 27 (정답은 21건)
  긴급 판단이 바뀐 건 5
  R006 실험 전 False → True · 정답 False · 출입 카드를 잃어버렸습니다. 정지 처리 부탁드립니다.
  R009 실험 전 False → True · 정답 False · 야간에 후문 출입 통제기가 열려 있었습니다. 보안 점검 부탁드
  R033 실험 전 False → True · 정답 False · 창고 C 셔터가 중간에서 멈췄습니다. 자재를 넣고 빼지 못하고
  R061 실험 전 False → True · 정답 False · 주차장 차단기가 올라가지 않아 출근 차량이 밀려 있습니다.
  R077 실험 전 False → True · 정답 False · 화재 감지기가 오작동으로 계속 울립니다. 사무동 2층입니다. 
